In [1]:
import sys
sys.path.append('RAG_TECHNIQUES')

In [3]:
import os
import sys
from dotenv import load_dotenv
from langchain.docstore.document import Document

from typing import List
# from rank_bm25 import BM25Okapi
import numpy as np


# Original path append replaced for Colab compatibility
from helper_functions import *
from evaluation.evalute_rag import *

# Load environment variables from a .env file
load_dotenv()

# Set the OpenAI API key environment variable
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

/var/folders/7m/51cswqps6c30dzq0vq0cyrbh0000gn/T/ipykernel_59469/2673782790.py:12: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  from helper_functions import *


In [4]:
import os
from bs4 import BeautifulSoup
import glob

docs = []

html_files = glob.glob("docs.lammps.org/*.html")  # adjust path
html_files.sort()
for file in html_files:
  soup = BeautifulSoup(open(file), "html.parser")
  heading_section = soup.find("h1")
  restriction_section = soup.find("section", {"id": "restrictions"})
  restriction_text = "none"  # Default to 'none' if section not found
  if restriction_section:
      # Find all <p> tags inside this section (usually one)
      paragraphs = restriction_section.find_all("p")
      restriction_text = " ".join(p.get_text(strip=True) for p in paragraphs)

      docs.append({
          "command": heading_section.get_text(strip=True),
          "restriction": restriction_text
      })

In [5]:
def encode_data():
    """
    Encodes a PDF book into a vector store using OpenAI embeddings.

    Args:
        path: The path to the PDF file.
        chunk_size: The desired size of each text chunk.
        chunk_overlap: The amount of overlap between consecutive chunks.

    Returns:
        A FAISS vector store containing the encoded book content.
    """
    cleaned_texts = [Document(page_content=str(p)) for p in docs]
    # print(cleaned_texts)

    # Create embeddings (Tested with OpenAI and Amazon Bedrock)
    embeddings = get_langchain_embedding_provider(EmbeddingProvider.OPENAI)

    # Create vector store
    vectorstore = FAISS.from_documents(cleaned_texts, embeddings)

    return vectorstore

In [6]:
chunks_vector_store = encode_data()

In [7]:
chunks_query_retriever = chunks_vector_store.as_retriever(search_kwargs={"k": 2})

In [8]:
test_query = "What are the restriction for dihedral_style cosine command"
context = retrieve_context_per_question(test_query, chunks_query_retriever)
show_context(context)

/Users/vinaykumar/Desktop/WebCrawler/RAG_TECHNIQUES/helper_functions.py:144: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = chunks_query_retriever.get_relevant_documents(question)


Context 1:
{'command': 'dihedral_style cosine/squared/restricted command\uf0c1', 'restriction': 'This dihedral style can only be used if LAMMPS was built with the\nEXTRA-MOLECULE package.  See theBuild packagedoc page\nfor more info.'}


Context 2:
{'command': 'dihedral_style cosine/shift/exp command\uf0c1', 'restriction': 'This dihedral style can only be used if LAMMPS was built with the\nMOLECULE package.  See theBuild packagedoc\npage for more info.'}


